# ControlNet Training — Dong Ho Folk Paintings (Colab)

Train ControlNet-scribble: hand-drawn sketch → Dong Ho-style painting.

## Prerequisites (làm local TRƯỚC khi chạy notebook)

1. Zip dataset đã augment:
   ```bash
   cd ~/Desktop/ControlNet/ControlNet/training && zip -r dongho.zip dongho/
   ```
2. Upload `dongho.zip` lên Google Drive root (`MyDrive/dongho.zip`).
3. Trong Colab: **Runtime → Change runtime type → GPU (T4)**.
4. Chạy các cell theo thứ tự.

## Notebook làm gì
- Clone ControlNet repo, mount Drive, giải nén dataset.
- Tải SD 1.5 base (~4 GB), tạo `control_sd15_ini.ckpt` qua `tool_add_control.py`.
- Patch `MyDataset` trỏ vào dữ liệu Dong Ho + augment scribble.
- Train batch=1 + grad-accum=4 + fp16 (vừa T4 16 GB).
- Lưu checkpoint định kỳ về Drive.

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Clone repo + cài deps

In [ ]:
%cd /content
!rm -rf ControlNet
!git clone https://github.com/lllyasviel/ControlNet.git
%cd /content/ControlNet

In [ ]:
!pip install -q "pytorch-lightning>=2.0,<3" omegaconf einops open_clip_torch \
  transformers timm safetensors ftfy opencv-python

### Patch ControlNet repo cho pytorch-lightning 2.x

PL 2.x bỏ `pytorch_lightning.utilities.distributed`. Cell dưới thay `rank_zero_only` bằng try/except để chạy được trên cả PL 1.x và 2.x.

In [ ]:
import pathlib
PATCH_OLD = 'from pytorch_lightning.utilities.distributed import rank_zero_only'
PATCH_NEW = ('try:\n'
             '    from pytorch_lightning.utilities.distributed import rank_zero_only\n'
             'except ImportError:\n'
             '    from pytorch_lightning.utilities.rank_zero import rank_zero_only')
for path in ['/content/ControlNet/cldm/logger.py',
             '/content/ControlNet/ldm/models/diffusion/ddpm.py']:
    p = pathlib.Path(path)
    s = p.read_text()
    if PATCH_OLD in s and PATCH_NEW not in s:
        p.write_text(s.replace(PATCH_OLD, PATCH_NEW))
        print('patched', path)
    else:
        print('skip   ', path)

## 3. Cấu hình hyperparameters

Sửa các giá trị dưới đây trước khi train. Train script ở cell ## 7 đọc các biến này qua env.

In [ ]:
TRAIN_CFG = {
    # ----- smoke test (test flow trước khi train full) -----
    'LIMIT_SAMPLES':    '500',     # 0 = dùng toàn bộ 1700 ảnh; >0 = chỉ lấy N ảnh đầu
    'MAX_STEPS':        '100',     # smoke test ngắn; full run đặt 5000+
    'LOGGER_FREQ':      '25',
    # ----- core hparams -----
    'BATCH_SIZE':       '1',
    'ACCUMULATE':       '4',
    'LEARNING_RATE':    '1e-5',
    'SD_LOCKED':        '1',       # 1=True an toàn; 0=False để học style mạnh (lr nên 2e-6)
    'ONLY_MID_CONTROL': '1',       # 1 nhanh hơn (T4); 0 giữ context tốt hơn (cần GPU mạnh)
    'RESOLUTION':       '384',
    'NUM_WORKERS':      '2',
}
print(TRAIN_CFG)

## 4. Mount Drive + giải nén dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!mkdir -p /content/ControlNet/training
!unzip -q /content/drive/MyDrive/dongho.zip -d /content/ControlNet/training/
!echo 'target:' && ls /content/ControlNet/training/dongho/target/ | wc -l
!echo 'source:' && ls /content/ControlNet/training/dongho/source/ | wc -l
!echo 'prompt.json lines:' && wc -l /content/ControlNet/training/dongho/prompt.json

## 5. Tải SD 1.5 base + tạo init checkpoint

In [ ]:
!mkdir -p /content/ControlNet/models
!wget -c https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-v1-5/resolve/main/v1-5-pruned.ckpt \
  -O /content/ControlNet/models/v1-5-pruned.ckpt

In [ ]:
%cd /content/ControlNet
import os, torch
from share import *
from cldm.model import create_model

INPUT  = './models/v1-5-pruned.ckpt'
OUTPUT = './models/control_sd15_ini.ckpt'

if os.path.exists(OUTPUT) and os.path.getsize(OUTPUT) > 5e9:
    print('Đã có', OUTPUT, f'({os.path.getsize(OUTPUT)/1e9:.2f} GB) — bỏ qua')
else:
    if os.path.exists(OUTPUT):
        os.remove(OUTPUT)
        print('Xoá file dở dang')

    print('[1/4] Tạo model từ config...')
    model = create_model(config_path='./models/cldm_v15.yaml')

    print('[2/4] Load SD weights (~4 GB, có thể mất 30–60s)...')
    pretrained = torch.load(INPUT, map_location='cpu', weights_only=False)
    if 'state_dict' in pretrained:
        pretrained = pretrained['state_dict']

    print('[3/4] Build ControlNet init state_dict...')
    scratch = model.state_dict()
    target = {}
    for k in scratch.keys():
        if k.startswith('control_'):
            copy_k = 'model.diffusion_' + k[len('control_'):]
        else:
            copy_k = k
        target[k] = (pretrained[copy_k] if copy_k in pretrained else scratch[k]).clone()
    model.load_state_dict(target, strict=True)

    print('[4/4] Saving ~5.5 GB (1–2 phút)...')
    torch.save(model.state_dict(), OUTPUT)
    print('Done →', OUTPUT, f'({os.path.getsize(OUTPUT)/1e9:.2f} GB)')

## 6. Patch dataset loader (đọc RESOLUTION từ TRAIN_CFG)

In [ ]:
from pathlib import Path
RES = int(TRAIN_CFG['RESOLUTION'])
Path('/content/ControlNet/tutorial_dataset.py').write_text(f'''import json
import random
import cv2
import numpy as np
from torch.utils.data import Dataset

DATA_ROOT = './training/dongho'
RESOLUTION = {RES}
AUGMENT_SCRIBBLE = True


def augment_scribble(rgb):
    """Randomly degrade a clean trace so training sources mimic rough hand sketches."""
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    h, w = gray.shape

    if random.random() < 0.6:
        it = random.randint(1, 2)
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
        op = cv2.dilate if random.random() < 0.5 else cv2.erode
        gray = op(gray, k, iterations=it)

    if random.random() < 0.4:
        mask = np.full_like(gray, 255)
        for _ in range(random.randint(2, 8)):
            x = random.randint(0, w - 1)
            y = random.randint(0, h - 1)
            r = random.randint(int(min(h, w) * 0.02), int(min(h, w) * 0.08))
            cv2.circle(mask, (x, y), r, 0, -1)
        gray = cv2.bitwise_and(gray, mask)

    if random.random() < 0.3:
        angle = random.uniform(-3, 3)
        M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
        gray = cv2.warpAffine(gray, M, (w, h), flags=cv2.INTER_NEAREST, borderValue=0)

    return cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)


class MyDataset(Dataset):
    def __init__(self):
        self.data = []
        with open(f"{{DATA_ROOT}}/prompt.json", "rt") as f:
            for line in f:
                self.data.append(json.loads(line))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        source = cv2.imread(f"{{DATA_ROOT}}/" + item["source"])
        target = cv2.imread(f"{{DATA_ROOT}}/" + item["target"])
        source = cv2.cvtColor(source, cv2.COLOR_BGR2RGB)
        target = cv2.cvtColor(target, cv2.COLOR_BGR2RGB)
        source = cv2.resize(source, (RESOLUTION, RESOLUTION), interpolation=cv2.INTER_AREA)
        target = cv2.resize(target, (RESOLUTION, RESOLUTION), interpolation=cv2.INTER_AREA)

        if AUGMENT_SCRIBBLE:
            source = augment_scribble(source)

        source = source.astype(np.float32) / 255.0
        target = (target.astype(np.float32) / 127.5) - 1.0
        return dict(jpg=target, txt=item["prompt"], hint=source)
''')
print('Đã ghi tutorial_dataset.py với RESOLUTION =', RES)

## 7. Training script (đọc config qua env vars)

In [ ]:
from pathlib import Path
Path('/content/ControlNet/train_dongho.py').write_text('''import os
from share import *

import torch
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint
from torch.utils.data import DataLoader, Subset

from tutorial_dataset import MyDataset
from cldm.logger import ImageLogger
from cldm.model import create_model, load_state_dict

resume_path      = "./models/control_sd15_ini.ckpt"
batch_size       = int(os.environ.get("BATCH_SIZE", 1))
accumulate       = int(os.environ.get("ACCUMULATE", 4))
max_steps        = int(os.environ.get("MAX_STEPS", 5000))
logger_freq      = int(os.environ.get("LOGGER_FREQ", 500))
learning_rate    = float(os.environ.get("LEARNING_RATE", 1e-5))
sd_locked        = bool(int(os.environ.get("SD_LOCKED", 1)))
only_mid_control = bool(int(os.environ.get("ONLY_MID_CONTROL", 1)))
num_workers      = int(os.environ.get("NUM_WORKERS", 2))
limit_samples    = int(os.environ.get("LIMIT_SAMPLES", 0))

print(f"cfg: bs={batch_size} accum={accumulate} steps={max_steps} "
      f"lr={learning_rate} sd_locked={sd_locked} only_mid={only_mid_control} "
      f"limit={limit_samples}")

model = create_model("./models/cldm_v15.yaml").cpu()
model.load_state_dict(load_state_dict(resume_path, location="cpu"))
model.learning_rate = learning_rate
model.sd_locked = sd_locked
model.only_mid_control = only_mid_control

dataset = MyDataset()
if limit_samples > 0:
    n = min(limit_samples, len(dataset))
    dataset = Subset(dataset, range(n))
    print(f"[smoke] dùng {n}/{limit_samples} ảnh đầu")
else:
    print(f"dùng toàn bộ {len(dataset)} ảnh")

dataloader = DataLoader(dataset, num_workers=num_workers, batch_size=batch_size, shuffle=True)

image_logger = ImageLogger(batch_frequency=logger_freq)
ckpt_cb = ModelCheckpoint(
    dirpath="./training/dongho_ckpts",
    filename="dongho-{step:06d}",
    every_n_train_steps=max(50, max_steps // 5),
    save_top_k=-1,
    save_last=True,
)

if torch.cuda.is_available():
    accelerator, devices, precision = "gpu", 1, 16
    print("Train on GPU:", torch.cuda.get_device_name(0))
else:
    accelerator, devices, precision = "cpu", 1, 32
    print("WARNING: GPU không có sẵn — train trên CPU sẽ rất chậm.")

trainer = pl.Trainer(
    accelerator=accelerator,
    devices=devices,
    precision=precision,
    accumulate_grad_batches=accumulate,
    max_steps=max_steps,
    callbacks=[image_logger, ckpt_cb],
    default_root_dir="./training/dongho_ckpts",
)

trainer.fit(model, dataloader)
''')
print('Đã ghi train_dongho.py')

## 8. Train

Mặc định: smoke-test 500 ảnh × 100 step (~5–10 phút trên T4) để xác nhận flow chạy không lỗi.
Sau khi smoke-test xong, vào cell `## 3` đặt `LIMIT_SAMPLES='0'` + `MAX_STEPS='5000'` rồi chạy lại từ cell `## 7`.

Nếu OOM: giảm `RESOLUTION`/`MAX_STEPS`, hoặc giữ `ONLY_MID_CONTROL='1'`.

In [ ]:
import os, subprocess, sys
env = {**os.environ, **TRAIN_CFG}
proc = subprocess.Popen([sys.executable, 'train_dongho.py'],
                        cwd='/content/ControlNet',
                        env=env,
                        stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT,
                        text=True,
                        bufsize=1)
try:
    for line in proc.stdout:
        print(line, end='')
    proc.wait()
except KeyboardInterrupt:
    proc.terminate()
    proc.wait()
    print('\n[Interrupted]')
print('Exit code:', proc.returncode)

## 9. Backup checkpoints về Drive

In [ ]:
!mkdir -p /content/drive/MyDrive/dongho_ckpts
!cp -n /content/ControlNet/training/dongho_ckpts/*.ckpt /content/drive/MyDrive/dongho_ckpts/ || true
!ls -lh /content/drive/MyDrive/dongho_ckpts/

## 10. (Tuỳ chọn) Inference test

Sau khi train xong, test với một sketch (nét đen trên nền trắng):

In [ ]:
import os, cv2, numpy as np, torch, einops
from pytorch_lightning import seed_everything
from cldm.model import create_model, load_state_dict
from cldm.ddim_hacked import DDIMSampler

CKPT = '/content/ControlNet/training/dongho_ckpts/last.ckpt'
SKETCH_PATH = '/content/my_sketch.png'
USER_INPUT_IS_BLACK_ON_WHITE = True
RES = int(TRAIN_CFG['RESOLUTION'])
PROMPT   = 'Vietnamese Dong Ho folk woodblock painting, warrior riding a beast'
A_PROMPT = 'best quality, extremely detailed, traditional woodblock print'
N_PROMPT = 'blurry, low quality, deformed'

assert os.path.exists(CKPT), (
    f'Không thấy {CKPT}. Bạn cần chạy cell ## 8 (Train) đến khi xong rồi mới chạy cell này. '
    'Hoặc liệt kê checkpoint đã có: !ls /content/ControlNet/training/dongho_ckpts/'
)
assert os.path.exists(SKETCH_PATH), (
    f'Không thấy sketch {SKETCH_PATH}. Upload ảnh sketch (nét đen trên nền trắng) lên Colab '
    'qua Files panel rồi chỉnh biến SKETCH_PATH.'
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = create_model('/content/ControlNet/models/cldm_v15.yaml').cpu()
model.load_state_dict(load_state_dict(CKPT, location=device))
model = model.to(device)
sampler = DDIMSampler(model)

sketch = cv2.cvtColor(cv2.imread(SKETCH_PATH), cv2.COLOR_BGR2RGB)
sketch = cv2.resize(sketch, (RES, RES))
gray = cv2.cvtColor(sketch, cv2.COLOR_RGB2GRAY)
_, mask = cv2.threshold(gray, 127, 255,
                        cv2.THRESH_BINARY_INV if USER_INPUT_IS_BLACK_ON_WHITE else cv2.THRESH_BINARY)
scribble = cv2.cvtColor(mask, cv2.COLOR_GRAY2RGB)

with torch.no_grad():
    control = torch.from_numpy(scribble.copy()).float().to(device) / 255.0
    control = einops.rearrange(control, 'h w c -> 1 c h w')
    seed_everything(42)
    cond    = {'c_concat': [control], 'c_crossattn': [model.get_learned_conditioning([PROMPT + ', ' + A_PROMPT])]}
    un_cond = {'c_concat': [control], 'c_crossattn': [model.get_learned_conditioning([N_PROMPT])]}
    shape = (4, RES // 8, RES // 8)
    model.control_scales = [1.0] * 13
    samples, _ = sampler.sample(20, 1, shape, cond, verbose=False, eta=0.0,
                                unconditional_guidance_scale=9.0,
                                unconditional_conditioning=un_cond)
    x = model.decode_first_stage(samples)
    x = (einops.rearrange(x, 'b c h w -> b h w c') * 127.5 + 127.5).cpu().numpy().clip(0, 255).astype(np.uint8)

from PIL import Image
Image.fromarray(x[0]).save('/content/test_output.png')
print('Saved to /content/test_output.png')
from IPython.display import Image as IPImg
IPImg('/content/test_output.png')